# 02: Processes and Environment

When you click "Run" on a code cell, something happens on your computer. When Claude Code executes a command, something happens. But what, exactly?

This notebook explains **processes** (programs that are running) and the **environment** (the configuration they inherit). Understanding these is the key to debugging "it works on my machine" problems and understanding why things like virtual environments and API keys work the way they do.

> **References:**
> - [MIT Missing Semester — Lecture 1: The Shell](https://missing.csail.mit.edu/2020/course-shell/) — covers processes, environment variables, and how the shell works
> - [MIT Missing Semester — Lecture 3: Editors (vim)](https://missing.csail.mit.edu/2020/editors/) — touches on process management and job control
> - [MIT Missing Semester — Lecture 4: Data Wrangling](https://missing.csail.mit.edu/2020/data-wrangling/) — pipes, redirection, and combining commands

> **Navigation:** [← Previous: The Filesystem and the Shell](../02-your-computer/01-filesystem-and-shell.ipynb) | [Module Index](../README.md) | [Next: Networking and APIs →](../02-your-computer/03-networking-and-apis.ipynb)

> **Why this matters for your work**
> 
> - Your `ANTHROPIC_API_KEY` is stored as an environment variable. If you don't understand environment variables, you can't set up API access, debug "authentication failed" errors, or keep your key secure.
> - Virtual environments are why `pip install` for one project doesn't break another. When you're running RFdiffusion in one environment and your data analysis in another, venvs keep them isolated. Understanding this prevents the nightmare of conflicting package versions.
> - When a Python script "hangs" or Claude Code seems stuck, understanding processes tells you what's actually running and how to fix it. Every command Claude Code executes is a process — knowing that lets you understand timeouts, errors, and resource usage.
> - Pipes and stdout/stderr are how Claude Code captures output from commands it runs. When you see Claude Code showing you the results of `python analysis.py`, it's reading that process's stdout.

---

## What is a process?

A **process** is a running instance of a program. When you:
- Open VS Code → that's a process
- Run a Python script → that's a process
- Open Terminal → the shell (zsh) is a process
- Run a Jupyter notebook → the kernel is a process
- Use Claude Code → it spawns processes for every command it runs

Right now, your computer is running hundreds of processes. Let's look.

In [ ]:
import os

# Every process gets a unique ID number (PID)
my_pid = os.getpid()
print(f"This Python process has PID: {my_pid}")

# Every process also has a parent process
parent_pid = os.getppid()
print(f"Its parent process has PID: {parent_pid}")
print()
print("The parent is Jupyter — it started this Python kernel.")
print("Jupyter was started by your shell. Your shell was started by Terminal/VS Code.")
print("It's processes all the way down.")

In [ ]:
# Let's see what's running on your Mac right now
# ps aux lists all processes; we'll just grab a sample
import subprocess

result = subprocess.run(["ps", "aux"], capture_output=True, text=True)
lines = result.stdout.strip().split("\n")

print(f"Total processes running: {len(lines) - 1}")  # minus the header
print()
print("Header:")
print(lines[0])
print()
print("First 5 processes:")
for line in lines[1:6]:
    print(line)

In [ ]:
# Find Python-related processes specifically
python_processes = [line for line in lines if "python" in line.lower() or "jupyter" in line.lower()]

print(f"Python/Jupyter processes: {len(python_processes)}")
for proc in python_processes:
    print(proc[:120])  # truncate long lines

---

## What happens when you "run" something

Here's the sequence when you press `Shift+Enter` on a code cell:

1. VS Code sends the cell's code to the **Jupyter kernel** (a running Python process)
2. The kernel **executes** the code
3. Any output (from `print()`, return values, etc.) gets sent back to VS Code
4. VS Code **displays** the output below the cell

When Claude Code runs a shell command:

1. Claude Code tells your shell (zsh) to run the command
2. The shell **creates a new process** for that command
3. The command runs and produces output
4. Claude Code **reads the output** and uses it to decide what to do next

The key insight: **every command is a process**. It starts, does its work, and ends.

```mermaid
graph TD
    subgraph "What happens when you press Shift+Enter on a code cell"
        A["You press <b>Shift+Enter</b><br/>in VS Code"] --> B["VS Code sends cell code to<br/>the <b>Jupyter kernel</b><br/>(a running Python process)"]
        B --> C["The kernel <b>executes</b><br/>your Python code"]
        C --> D["Output from print() and<br/>return values go to <b>stdout</b>"]
        D --> E["VS Code <b>displays</b> the output<br/>below the cell"]
    end

    subgraph "What happens when Claude Code runs a command"
        F["Claude Code decides to run<br/>e.g. <code>pip install pandas</code>"] --> G["Your <b>shell</b> (zsh) creates<br/>a <b>new process</b>"]
        G --> H["The process <b>inherits</b><br/>environment variables<br/>(PATH, API keys, etc.)"]
        H --> I["Command runs, writes to<br/><b>stdout</b> and <b>stderr</b>"]
        I --> J["Claude Code <b>reads output</b><br/>and decides next step"]
    end

    style A fill:#4488cc,color:#fff
    style F fill:#44aa88,color:#fff
    style E fill:#4488cc,color:#fff
    style J fill:#44aa88,color:#fff
```

In [ ]:
# We can create processes from Python too — this is what Claude Code does
import subprocess

# Run 'ls' as a separate process
result = subprocess.run(
    ["ls", "-la", str(Path.home() / "ai-tutorial")],
    capture_output=True,  # capture stdout and stderr
    text=True             # return strings, not bytes
)

print("=== stdout (normal output) ===")
print(result.stdout[:500])
print(f"\n=== Return code: {result.returncode} ===")
print("(0 means success, anything else means error)")

---

## Environment variables

Every process inherits a set of **environment variables** — key-value pairs that configure how things work. Think of them like lab-wide settings posted on the wall: any protocol can reference them.

You already use one: `ANTHROPIC_API_KEY`. Instead of pasting your API key into every script, you set it as an environment variable once, and any program that needs it can read it.

That's the whole point: **separate configuration from code**.

In [ ]:
import os

# See ALL environment variables
# (There are a lot — let's just count them and show a few)
all_vars = dict(os.environ)
print(f"Total environment variables: {len(all_vars)}")
print()

# Some important ones
interesting_vars = ["HOME", "USER", "SHELL", "PATH", "TERM", "LANG"]
for var in interesting_vars:
    value = os.environ.get(var, "(not set)")
    # PATH is super long, so truncate it
    if var == "PATH":
        value = value[:80] + "..."
    print(f"{var:12s} = {value}")

In [ ]:
# Check if your Anthropic API key is set
# (We won't print the actual key — that's a secret!)
api_key = os.environ.get("ANTHROPIC_API_KEY")

if api_key:
    # Show just enough to confirm it's there
    print(f"ANTHROPIC_API_KEY is set: {api_key[:8]}...{api_key[-4:]}")
    print(f"Length: {len(api_key)} characters")
else:
    print("ANTHROPIC_API_KEY is NOT set.")
    print("You'll need this for the API modules later.")
    print("It's typically set in your ~/.zshrc file.")

```mermaid
graph LR
    subgraph "How ANTHROPIC_API_KEY flows from config to API call"
        A["<b>~/.zshrc</b><br/><code>export ANTHROPIC_API_KEY=sk-ant-...</code>"] -->|"loaded on<br/>terminal open"| B["<b>Shell (zsh)</b><br/>Key is now in<br/>shell environment"]
        B -->|"inherited by<br/>child process"| C["<b>Python process</b><br/><code>os.environ['ANTHROPIC_API_KEY']</code>"]
        C -->|"read by<br/>anthropic library"| D["<b>API Request</b><br/>Key sent as HTTP header<br/><code>x-api-key: sk-ant-...</code>"]
        D -->|"over HTTPS"| E["<b>Anthropic Server</b><br/>Authenticates you,<br/>processes prompt"]
    end

    style A fill:#cc8844,color:#fff
    style B fill:#4488cc,color:#fff
    style C fill:#44aa88,color:#fff
    style D fill:#aa44aa,color:#fff
    style E fill:#cc4444,color:#fff
```

This is why you set the key in `~/.zshrc` once and it "just works" everywhere -- environment variable inheritance passes it from parent process to child process automatically.

### Where do environment variables come from?

On your Mac, environment variables are typically set in:

1. **`~/.zshrc`** — your shell's config file, loaded every time you open a terminal
2. **`~/.zprofile`** — loaded on login (similar to `.zshrc` but runs less often)
3. **`.env` files** — project-specific variables (used by some tools)
4. **Inline** — `ANTHROPIC_API_KEY=sk-... python script.py` (temporary, just for that command)

When you added your `ANTHROPIC_API_KEY` to `~/.zshrc`, here's what happens:
1. You open a terminal → zsh starts → it reads `~/.zshrc` → your key is now in the environment
2. You run `python` → the Python process **inherits** all env vars from the shell
3. Your Python code calls `os.environ["ANTHROPIC_API_KEY"]` → it finds the key

It's inheritance: parent passes environment to child.

> **Why PATH matters for you:** When you type `python` and get the wrong version, or `pip install` puts packages somewhere your scripts can't find them, it's a PATH problem. Understanding PATH is how you debug "command not found" and "module not found" errors that block your analysis work.

---

## PATH: how your shell finds programs

When you type `python` in the terminal, how does your Mac know where the Python program actually lives? The answer is the **PATH** environment variable.

PATH is a list of directories, separated by colons. When you type a command, your shell searches through these directories in order until it finds a matching program.

In [ ]:
import os

# Let's look at your PATH, one directory per line
path_dirs = os.environ["PATH"].split(":")

print(f"Your PATH has {len(path_dirs)} directories:")
print()
for i, directory in enumerate(path_dirs, 1):
    print(f"  {i:2d}. {directory}")

In [ ]:
# 'which' tells you which program the shell will use
import shutil

programs_to_find = ["python", "python3", "pip", "jupyter", "git", "code", "claude"]

print("Where key programs live:")
for prog in programs_to_find:
    location = shutil.which(prog)
    if location:
        print(f"  {prog:12s} → {location}")
    else:
        print(f"  {prog:12s} → (not found in PATH)")

Notice how some programs live in `.venv/bin/` — that's the virtual environment at work.

---

## Virtual environments demystified

A Python **virtual environment** (venv) is one of those things that seems magical until you understand it. Here's the trick: **it's just a directory with its own copy of Python and packages, and an adjustment to PATH.**

When you "activate" a virtual environment, all that happens is:
1. The venv's `bin/` directory gets added to the **front** of your PATH
2. So when you type `python`, your shell finds the venv's Python **first**
3. When you `pip install` something, it goes into the venv's `lib/` directory

That's it. No magic. Just PATH manipulation.

In [ ]:
from pathlib import Path
import sys

# Where is the Python running THIS notebook?
print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version}")
print()

# Is it inside a virtual environment?
venv_path = Path.home() / "ai-tutorial" / ".venv"
if venv_path.exists():
    print(f"Virtual environment exists at: {venv_path}")
    print()
    
    # What's inside the venv?
    print("Contents of .venv/:")
    for item in sorted(venv_path.iterdir()):
        print(f"  {item.name}/" if item.is_dir() else f"  {item.name}")

In [ ]:
# What packages are installed in our environment?
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=columns"],
    capture_output=True, text=True
)

lines = result.stdout.strip().split("\n")
print(f"Installed packages: {len(lines) - 2}")  # minus header lines
print()
# Show just a few key ones
key_packages = ["jupyter", "pandas", "numpy", "matplotlib", "anthropic", "seaborn"]
for line in lines:
    if any(pkg in line.lower() for pkg in key_packages):
        print(f"  {line.strip()}")

### Why virtual environments matter

Imagine you have two projects:
- **Pain imaging analysis** needs pandas 2.0 and numpy 1.26
- **Old RNA-seq pipeline** needs pandas 1.5 and numpy 1.24

Without virtual environments, installing packages for one project would break the other. Each venv is an isolated bubble — its own Python, its own packages, no conflicts.

---

## Stdin, stdout, stderr — what `print()` actually does

Every process has three standard **streams** (channels for data):

| Stream | Name | What it's for | Python example |
|--------|------|---------------|----------------|
| **stdin** | Standard input | Data coming IN to the program | `input("Enter value: ")` |
| **stdout** | Standard output | Normal output | `print("Hello")` |
| **stderr** | Standard error | Error messages, warnings | Error tracebacks |

When you call `print("Hello")`, Python writes "Hello" to **stdout**. The terminal (or Jupyter) picks it up and shows it to you.

When Python hits an error, the traceback goes to **stderr** — a separate channel. This separation matters because you can redirect them independently.

In [ ]:
import sys

# Normal output goes to stdout
print("This is stdout (normal output)")

# You can also write to stderr explicitly
print("This is stderr (error channel)", file=sys.stderr)

# Both appear in the notebook, but in a real terminal they're separate streams

In [ ]:
# When Claude Code runs a command, it captures both stdout and stderr separately
import subprocess

# This command succeeds — output goes to stdout
result = subprocess.run(["ls", "/tmp"], capture_output=True, text=True)
print(f"stdout has {len(result.stdout)} characters")
print(f"stderr has {len(result.stderr)} characters")
print(f"return code: {result.returncode}")
print()

# This command fails — error goes to stderr
result = subprocess.run(["ls", "/nonexistent_directory"], capture_output=True, text=True)
print(f"stdout has {len(result.stdout)} characters")
print(f"stderr has {len(result.stderr)} characters")
print(f"stderr says: {result.stderr.strip()}")
print(f"return code: {result.returncode}")

---

## Pipes and redirection

In the shell, you can connect processes together like plumbing:

- **Pipe** (`|`): send one command's stdout into another command's stdin
  ```bash
  cat data.csv | head -5      # read file, then show first 5 lines
  ps aux | grep python         # list processes, then filter for "python"
  ```

- **Redirect** (`>`, `>>`): send stdout to a file instead of the screen
  ```bash
  ls -la > file_list.txt       # write output to file (overwrites)
  echo "new line" >> log.txt   # append to file
  ```

- **Redirect stderr** (`2>`): send errors to a file
  ```bash
  python script.py 2> errors.log   # errors go to file, output to screen
  ```

You'll see Claude Code use pipes all the time — for example, `grep -r "TRPV1" . | head -20` to search for a gene name and show just the first 20 matches.

> **See also:** [MIT Missing Semester — Lecture 4: Data Wrangling](https://missing.csail.mit.edu/2020/data-wrangling/) goes deep on pipes and data manipulation.

In [ ]:
# Pipes in the shell
# This is how you'd search for Python processes in the terminal:
# ps aux | grep python

# In Jupyter, we can use the ! prefix:
!ps aux | grep -i python | head -5

In [ ]:
# The Python equivalent of piping: subprocess with shell=True
import subprocess

# Count how many .ipynb files are in the project
result = subprocess.run(
    "find ~/ai-tutorial -name '*.ipynb' | wc -l",
    shell=True, capture_output=True, text=True
)
print(f"Number of notebooks in the project: {result.stdout.strip()}")

---

## Putting it all together: what Claude Code actually does

When Claude Code runs a command on your behalf, here's the full picture:

1. **Claude Code** (a process) asks your **shell** (zsh, another process) to run a command
2. The shell **creates a new process** for that command
3. The new process **inherits** the shell's environment variables (including `ANTHROPIC_API_KEY`, `PATH`, etc.)
4. The command runs, writing to **stdout** (normal output) and **stderr** (errors)
5. Claude Code **captures** both streams
6. The process **exits** with a return code (0 = success)
7. Claude Code reads the output and decides what to do next

Now when you see Claude Code output like:
```
Running: pip install pandas
```
You know exactly what's happening under the hood.

---

## Exercises

### Exercise 1: Inspect your environment

Write code that:
1. Prints the value of `HOME`, `USER`, and `SHELL` environment variables
2. Checks whether `ANTHROPIC_API_KEY` is set (print "set" or "not set" — don't print the actual key)
3. Counts how many directories are in your PATH

In [ ]:
import os

# Your code here
for var in ["HOME", "USER", "SHELL"]:
    print(f"{var} = {os.environ.get(var, '(not set)')}")

print()
api_key_status = "set" if os.environ.get("ANTHROPIC_API_KEY") else "not set"
print(f"ANTHROPIC_API_KEY: {api_key_status}")

print()
path_count = len(os.environ["PATH"].split(":"))
print(f"PATH contains {path_count} directories")

### Exercise 2: Find where things live

Use `shutil.which()` to find the location of these programs: `python3`, `git`, `pip`, `ls`, `zsh`. For each one, also check whether it's inside the `.venv` directory.

In [ ]:
import shutil

# Your code here
programs = ["python3", "git", "pip", "ls", "zsh"]

for prog in programs:
    location = shutil.which(prog)
    if location:
        in_venv = ".venv" in location
        venv_note = " (in virtual environment)" if in_venv else ""
        print(f"{prog:10s} → {location}{venv_note}")
    else:
        print(f"{prog:10s} → not found")

### Exercise 3: Run a command and inspect the result

Use `subprocess.run()` to run `git log --oneline -5` in the `ai-tutorial` directory. Capture the output and print:
- The return code
- The number of lines of output
- The output itself

In [ ]:
import subprocess
from pathlib import Path

# Your code here
result = subprocess.run(
    ["git", "log", "--oneline", "-5"],
    capture_output=True, text=True,
    cwd=str(Path.home() / "ai-tutorial")
)

print(f"Return code: {result.returncode}")
lines = result.stdout.strip().split("\n")
print(f"Lines of output: {len(lines)}")
print()
print("Recent commits:")
print(result.stdout)

---

## What you just learned

- A **process** is a running instance of a program — every command is a process
- Processes have a **PID** (process ID) and a **parent** process
- **Environment variables** are key-value pairs that configure processes (like `ANTHROPIC_API_KEY`)
- **PATH** tells your shell where to find programs — that's how `python` resolves to an actual file
- A **virtual environment** is just a directory with its own Python + packages, activated by modifying PATH
- Processes have three streams: **stdin** (input), **stdout** (output), **stderr** (errors)
- **Pipes** (`|`) connect one process's output to another's input
- When Claude Code runs commands, it's creating processes and reading their stdout/stderr

Next up: what happens when your code talks to the internet — networking and APIs.

---

## Further Reading

- **Missing Semester, Lecture 1**: [The Shell](https://missing.csail.mit.edu/2020/course-shell/) — foundational lecture covering processes, environment variables, PATH, and shell basics
- **Missing Semester, Lecture 3**: [Editors (vim)](https://missing.csail.mit.edu/2020/editors/) — includes process management, job control (`Ctrl-Z`, `bg`, `fg`), and signals
- [Python docs: `os` module](https://docs.python.org/3/library/os.html) — the official reference for `os.environ`, `os.getpid()`, `os.getppid()`, and other OS-level functions used in this notebook
- [Python docs: `subprocess` module](https://docs.python.org/3/library/subprocess.html) — the official reference for `subprocess.run()` and process creation from Python
- [Real Python: Environment Variables in Python](https://realpython.com/python-environment-variables/) — a practical guide to reading, setting, and using environment variables with `os.environ` and `.env` files
- [Python docs: `venv`](https://docs.python.org/3/library/venv.html) — official documentation for virtual environments, explaining the PATH manipulation described in this notebook
- [Python docs: `sys` module](https://docs.python.org/3/library/sys.html) — reference for `sys.executable`, `sys.version`, `sys.path`, and other interpreter-level attributes

> **Navigation:** [← Previous: The Filesystem and the Shell](../02-your-computer/01-filesystem-and-shell.ipynb) | [Module Index](../README.md) | [Next: Networking and APIs →](../02-your-computer/03-networking-and-apis.ipynb)

---

## Edit Log

- 2026-03-25: Created notebook with initial content
- 2026-03-25: Added "Why this matters" rationale section
- 2026-03-25: Added external references and Further Reading section
- 2026-03-25: Added cross-module navigation links